## U-net

En este notebook se presenta el proceso de construccion del U-net autoencoder, y su respectivo entrenamiento y validacion. El objetivo principal es entrenar el modelo con distintas funciones de pérdida y evaluar
su capacidad para reconstruir imágenes normales con alta fidelidad, mientras genera
errores notorios en imágenes con defectos, lo que permite usarlo como detector de anomalías
sin supervisión.

In [ ]:
!python -m pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.4/848.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.2/852.2 kB 32.2 MB/s eta 0:00:00


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.functional import structural_similarity_index_measure as ssim # Preguntar al profe si se puede usar
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import lightning as L

## Creación modelo

### Encoder

El **encoder** reduce progresivamente la resolución espacial de la imagen mientras
aumenta los canales de características. Utiliza bloques de convoluciones con
`stride=2` (equivalente a max-pooling) para hacer el downsampling.

La característica clave del U-Net es que **todos los bloques intermedios guardan
skip connections** (conexiones residuales). Por ejemplo, con `hidden_channels = [32, 64, 128, 256]`:
- Los bloques `[32, 64, 128]` producen mapas de características que se guardan como skips
- El último bloque `[256]` produce el **bottleneck** — la representación más comprimida

A diferencia del VAE, este encoder es **determinístico**: no genera parámetros de
distribución (μ, log σ²), sino mapas de activación directos.

In [ ]:
class UNetEncoder(nn.Module):
    def __init__(
        self,
        in_channels=3,
        hidden_channels=None,
    ):
        super().__init__()

        if hidden_channels is None:
            hidden_channels = [32, 64, 128, 256]

        self.blocks = nn.ModuleList()

        current_channels = in_channels
        for out_ch in hidden_channels:
            self.blocks.append(
                nn.Sequential(
                    nn.Conv2d(current_channels, out_ch, kernel_size=4, stride=2, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                )
            )
            current_channels = out_ch

    def forward(self, x):
        skips = []
        h = x
        # Todos los bloques menos el último guardan skip connections
        # Ej: [32, 64, 128, 256] → skips en 32, 64, 128; bottleneck en 256
        for block in self.blocks[:-1]:
            h = block(h)
            skips.append(h)

        bottleneck = self.blocks[-1](h)
        return bottleneck, skips


#### Verificación del Encoder
Se instancia el encoder con los hiperparámetros predeterminados y se pasa un
batch de prueba para verificar que los shapes de salida sean correctos.

In [ ]:
encoder = UNetEncoder(
    in_channels=3,
    hidden_channels=[32, 64, 128, 256],
)

x = torch.randn(16, 3, 128, 128)

bottleneck, skips = encoder(x)

print(bottleneck.shape)
print(len(skips))
print(skips[0].shape)
print(skips[1].shape)
print(skips[2].shape)

torch.Size([16, 256, 8, 8])
3
torch.Size([16, 32, 64, 64])
torch.Size([16, 64, 32, 32])
torch.Size([16, 128, 16, 16])


### Decoder

El **decoder** reconstruye la imagen original a partir del bottleneck, usando
las skip connections del encoder para recuperar detalles espaciales perdidos.

El proceso en cada paso es:
1. **Upsampling** con `ConvTranspose2d` → duplica la resolución espacial
2. **Concatenación** con la skip connection correspondiente → duplica los canales
3. **Convolución de refinamiento** → combina la información del decoder y del skip

Este mecanismo es la gran ventaja del U-Net: en lugar de reconstruir todo desde
la representación comprimida, el decoder tiene acceso directo a mapas de
características de alta resolución del encoder, lo que resulta en reconstrucciones
mucho más nítidas.

La capa final aplica `ConvTranspose2d` para recuperar la resolución original
y una activación `Sigmoid` para normalizar los píxeles al rango [1].

In [ ]:
class UNetDecoder(nn.Module):
    def __init__(
        self,
        out_channels=3,
        hidden_channels=None,
        use_sigmoid=True,
    ):
        super().__init__()

        if hidden_channels is None:
            hidden_channels = [32, 64, 128, 256]

        # reversed: [256, 128, 64, 32]
        reversed_ch = list(reversed(hidden_channels))

        self.up_convs = nn.ModuleList()
        self.refine_convs = nn.ModuleList()

        # Cada paso: upsample + concat con skip + refinamiento
        # Ej con [256, 128, 64, 32]:
        #   i=0: bottleneck [B,256,8,8]  → up→[B,128,16,16] → cat skip[B,128,16,16] → refine→[B,128,16,16]
        #   i=1:            [B,128,16,16] → up→[B,64,32,32]  → cat skip[B,64,32,32]  → refine→[B,64,32,32]
        #   i=2:            [B,64,32,32]  → up→[B,32,64,64]  → cat skip[B,32,64,64]  → refine→[B,32,64,64]
        for i in range(len(reversed_ch) - 1):
            in_ch = reversed_ch[i]
            out_ch = reversed_ch[i + 1]

            self.up_convs.append(
                nn.Sequential(
                    nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                )
            )

            # Después de concatenar: out_ch (upsample) + out_ch (skip) = 2*out_ch entradas
            self.refine_convs.append(
                nn.Sequential(
                    nn.Conv2d(out_ch * 2, out_ch, kernel_size=3, stride=1, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                )
            )

        # Última capa: upsample a resolución original sin skip connection
        # reversed_ch[-1] → out_channels (ej: 32 → 3)
        final_layers = [
            nn.ConvTranspose2d(reversed_ch[-1], out_channels, kernel_size=4, stride=2, padding=1),
        ]
        if use_sigmoid:
            final_layers.append(nn.Sigmoid())

        self.final_conv = nn.Sequential(*final_layers)

    def forward(self, bottleneck, skips):
        h = bottleneck
        # Los skips vienen de superficial a profundo; los procesamos al revés
        reversed_skips = list(reversed(skips))

        for up_conv, refine_conv, skip in zip(self.up_convs, self.refine_convs, reversed_skips):
            h = up_conv(h)
            h = torch.cat([h, skip], dim=1)
            h = refine_conv(h)

        x_hat = self.final_conv(h)
        return x_hat

### UNetAutoEncoder (LightningModule)

Se define el modelo completo como un `LightningModule`, lo que permite separar
limpiamente la lógica del modelo del loop de entrenamiento.

#### Representación latente para t-SNE

Como el U-Net no tiene un espacio latente vectorial explícito (como el VAE), se
aplica **Global Average Pooling** sobre el bottleneck (B, C, H, W) → (B, C).
Este vector se usa para la visualización con t-SNE durante la validación.

#### Logging
El modelo registra automáticamente en WandB:
- Pérdida de train/val por época
- Grids de comparación de reconstrucción (validación y prueba)
- Visualización t-SNE del espacio latente por época

In [ ]:
class UNetAutoEncoder(L.LightningModule):
    def __init__(
        self,
        in_channels: int = 3,
        image_size: int = 128,
        hidden_channels: list[int] | None = None,
        lr: float = 1e-3,
        loss_type: str = "l1",
        use_sigmoid: bool = True,
    ):
        super().__init__()

        if hidden_channels is None:
            hidden_channels = [32, 64, 128, 256]

        self.save_hyperparameters()

        self.encoder = UNetEncoder(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
        )

        self.decoder = UNetDecoder(
            out_channels=in_channels,
            hidden_channels=hidden_channels,
            use_sigmoid=use_sigmoid,
        )

    def forward(self, x: torch.Tensor):
        bottleneck, skips = self.encoder(x)
        x_hat = self.decoder(bottleneck, skips)
        # Global average pooling del bottleneck como representación latente para t-SNE
        z = bottleneck.mean(dim=[2, 3])  # [B, C]
        return x_hat, z

    def reconstruction_loss(self, x_hat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        loss_type = self.hparams.loss_type.lower()

        if loss_type == "l1":
            return F.l1_loss(x_hat, x)

        if loss_type in ["l2", "mse"]:
            return F.mse_loss(x_hat, x)

        if loss_type == "ssim":
            return 1.0 - ssim(x_hat, x, data_range=1.0)

        if loss_type == "ssim+l1":
            ssim_loss = 1.0 - ssim(x_hat, x, data_range=1.0)
            l1_loss = F.l1_loss(x_hat, x)
            return ssim_loss + l1_loss

        raise ValueError(f"Loss no soportada: {self.hparams.loss_type}")

    def compute_loss(self, batch):
        x, _ = batch
        x_hat, z = self.forward(x)
        loss = self.reconstruction_loss(x_hat, x)
        return loss, x_hat, z

    def training_step(self, batch, batch_idx):
        loss, x_hat, z = self.compute_loss(batch)
        self.log("train/loss", loss, prog_bar=False, on_step=False, on_epoch=True)
        return loss

# --------------------------------------- Validation --------------------------------#

    def on_validation_epoch_start(self):
        self.val_z = []
        self.val_y = []
        self.val_x = []
        self.val_x_hat = []

    def validation_step(self, batch, batch_idx):
        loss, x_hat, z = self.compute_loss(batch)
        x, y = batch

        self.log("val/loss", loss, prog_bar=False, on_step=False, on_epoch=True)

        if len(self.val_x) < 2:
            self.val_x.append(x[:8].detach().cpu())
            self.val_x_hat.append(x_hat[:8].detach().cpu())

        self.val_z.append(z.detach().cpu())
        self.val_y.append(y.detach().cpu())

        return loss

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            self.val_z.clear()
            self.val_y.clear()
            self.val_x.clear()
            self.val_x_hat.clear()
            return

        if len(self.val_x) > 0 and len(self.val_x_hat) > 0:
            x = torch.cat(self.val_x, dim=0)
            x_hat = torch.cat(self.val_x_hat, dim=0)
            comparison = torch.cat([x, x_hat], dim=0)
            self._log_image_grid(
                key="val/reconstructions",
                images=comparison,
                caption="Fila 1: originales | Fila 2: reconstrucciones",
            )

        if len(self.val_z) > 0:
            z_all = torch.cat(self.val_z, dim=0)
            y_all = torch.cat(self.val_y, dim=0)
            self._log_tsne(z_all, y_all)

        self.val_z.clear()
        self.val_y.clear()
        self.val_x.clear()
        self.val_x_hat.clear()

# --------------------------------------- Testing --------------------------------#

    def on_test_epoch_start(self):
        self.test_good_x = []
        self.test_good_x_hat = []
        self.test_anom_x = []
        self.test_anom_x_hat = []

    def test_step(self, batch, batch_idx):
        loss, x_hat, z = self.compute_loss(batch)
        x, y = batch
        self.log("test/loss", loss, prog_bar=False, on_step=False, on_epoch=True)

        defect_code = y[:, 1]
        good_mask = defect_code == 0
        anom_mask = defect_code != 0

        if good_mask.any() and len(self.test_good_x) < 2:
            self.test_good_x.append(x[good_mask][:4].detach().cpu())
            self.test_good_x_hat.append(x_hat[good_mask][:4].detach().cpu())

        if anom_mask.any() and len(self.test_anom_x) < 2:
            self.test_anom_x.append(x[anom_mask][:4].detach().cpu())
            self.test_anom_x_hat.append(x_hat[anom_mask][:4].detach().cpu())

        return loss

    def on_test_epoch_end(self):
        if self.logger is None:
            return

        if len(self.test_good_x) > 0 and len(self.test_anom_x) > 0:
            good_x = torch.cat(self.test_good_x, dim=0)[:8]
            good_x_hat = torch.cat(self.test_good_x_hat, dim=0)[:8]
            anom_x = torch.cat(self.test_anom_x, dim=0)[:8]
            anom_x_hat = torch.cat(self.test_anom_x_hat, dim=0)[:8]

            comparison = torch.cat([good_x, good_x_hat, anom_x, anom_x_hat], dim=0)

            self._log_image_grid(
                key="test/good_vs_anomaly_reconstructions",
                images=comparison,
                caption=(
                    "Fila 1: good originales | "
                    "Fila 2: good reconstruidas | "
                    "Fila 3: anomalías originales | "
                    "Fila 4: anomalías reconstruidas"
                ),
                nrow=8,
            )

        self.test_good_x.clear()
        self.test_good_x_hat.clear()
        self.test_anom_x.clear()
        self.test_anom_x_hat.clear()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    ## ------------------ Funciones Auxiliares para logging -------------------------------- ##

    def _log_image_grid(self, key: str, images: torch.Tensor, caption: str = "", nrow: int = 16):
        if self.logger is None:
            return

        grid = torchvision.utils.make_grid(images, nrow=nrow, normalize=False)
        grid_np = grid.detach().cpu().permute(1, 2, 0).numpy()

        self.logger.experiment.log({
            key: wandb.Image(grid_np, caption=caption),
            "epoch": self.current_epoch,
        })

    def _log_tsne(self, z: torch.Tensor, labels: torch.Tensor):
        if self.logger is None:
            return

        z_np = z.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        if z_np.shape[0] < 3:
            return

        perplexity = min(30, max(2, z_np.shape[0] // 3))

        z_2d = TSNE(
            n_components=2,
            perplexity=perplexity,
            random_state=42,
            init="pca",
            learning_rate="auto",
        ).fit_transform(z_np)

        fig, ax = plt.subplots(figsize=(7, 6))
        scatter = ax.scatter(z_2d[:, 0], z_2d[:, 1], c=labels_np, s=12, alpha=0.8)
        ax.set_title(f"t-SNE del espacio latente - epoch {self.current_epoch}")
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
        fig.colorbar(scatter, ax=ax, label="Clase")
        fig.tight_layout()

        self.logger.experiment.log({
            "val/latent_tsne": wandb.Image(fig),
            "epoch": self.current_epoch,
        })

        plt.close(fig)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


Mounted at /content/drive/


In [ ]:
import torch
checkpoint_path = "/content/drive/MyDrive/Tarea3/cache/mvtec_anomaly_detection.pt"

checkpoint = torch.load(checkpoint_path, map_location="cpu")


In [ ]:
%cd /content
!git clone https://github.com/Chagui05/Tarea3-IA.git
%cd Tarea3-IA

/content
Cloning into 'Tarea3-IA'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 178 (delta 8), reused 22 (delta 8), pack-reused 153 (from 1)
Receiving objects: 100% (178/178), 60.09 MiB | 38.75 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/Tarea3-IA


In [ ]:
%cd /content/Tarea3-IA
!git fetch origin
!git checkout u-net

/content/Tarea3-IA
Branch 'u-net' set up to track remote branch 'u-net' from 'origin'.
Switched to a new branch 'u-net'


In [ ]:
!pip install lightning hydra-core wandb torchmetrics scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.5 MB/s eta 0:00:00


In [ ]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: davidblancolascarez (models-instituto-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
!python train.py \
  model=u-net \
  trainer.max_epochs=20 \
  data.checkpoint_path="/content/drive/MyDrive/Tarea3/cache/mvtec_anomaly_detection.pt" \
  trainer.accelerator=gpu \
  trainer.devices=1

data:
  checkpoint_path: /content/drive/MyDrive/Tarea3/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 1
model:
  name: u-net
  in_channels: 3
  image_size: 128
  hidden_channels:
  - 32
  - 64
  - 128
  - 256
  latent_dim: 256
  lr: 0.001
  loss_type: l1
  use_sigmoid: true
trainer:
  max_epochs: 20
  accelerator: gpu
  devices: 1
  check_val_every_n_epoch: 1
  log_every_n_steps: 1
  deterministic: true
  benchmark: false
logger:
  project: tarea-03-autoencoders
  name: ${model.name}_${model.loss_type}_z${model.latent_dim}
  log_model: false
seed: 42

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/proje